# Section 1: Load CCSR Features + Build Target Variable


In [5]:
import pandas as pd
import numpy as np
import duckdb
import warnings
warnings.filterwarnings('ignore')

# Load CCSR features built by teammate
# Shape: 9996 patients x 706 columns (subject_id + 705 CCSR flags)
ccsr = pd.read_parquet('outputs/ccsr_features.parquet')
print(f"CCSR features loaded: {ccsr.shape}")
print(f"First few columns: {list(ccsr.columns[:5])}")
print(ccsr.head())

CCSR features loaded: (9996, 706)
First few columns: ['subject_id', "'1    '", "'10   '", "'100  '", "'101  '"]
   subject_id  '1    '  '10   '  '100  '  '101  '  '102  '  '103  '  '104  '  \
0    19571305        0        0        0        0        0        0        0   
1    15403967        0        0        0        0        0        0        0   
2    10464873        0        0        0        0        0        0        0   
3    17535244        0        0        0        0        0        0        0   
4    17980127        0        0        0        0        0        0        0   

   '105  '  '106  '  ...  'SYM010'  'SYM011'  'SYM012'  'SYM013'  'SYM014'  \
0        0        0  ...         0         0         0         0         0   
1        0        0  ...         0         0         0         0         0   
2        0        0  ...         0         0         0         0         0   
3        0        0  ...         0         0         0         0         0   
4        0       

# Section 2: Build Target Variable and Join with CCSR Features


In [6]:

con = duckdb.connect()

# Build readmitted_30d target from admissions
target_query = """
WITH all_admissions AS (
    SELECT 
        a.subject_id,
        a.hadm_id,
        a.admittime,
        a.dischtime,
        a.admission_type,
        a.insurance,
        a.race,
        a.discharge_location,
        a.marital_status,
        p.gender,
        p.anchor_age + (
            EXTRACT(year FROM CAST(a.admittime AS TIMESTAMP)) 
            - p.anchor_year
        ) AS age_at_admission,
        DATE_DIFF('day',
            CAST(a.admittime AS TIMESTAMP),
            CAST(a.dischtime AS TIMESTAMP)
        ) AS los_days,
        CASE WHEN a.edregtime IS NOT NULL THEN 1 ELSE 0 END AS came_via_ed,
        ROW_NUMBER() OVER (
            PARTITION BY a.subject_id ORDER BY a.admittime
        ) - 1 AS prior_admissions_count,
        LEAD(a.admittime) OVER (
            PARTITION BY a.subject_id ORDER BY a.admittime
        ) AS next_admittime
    FROM 'data/admissions.csv' a
    JOIN 'data/patients.csv' p ON a.subject_id = p.subject_id
    WHERE a.subject_id IN (
        SELECT CAST(subject_id AS BIGINT) 
        FROM 'outputs/cohort_10k.csv'
    )
    AND a.hospital_expire_flag = 0
),
with_target AS (
    SELECT *,
        DATE_DIFF('day',
            CAST(dischtime AS TIMESTAMP),
            CAST(next_admittime AS TIMESTAMP)
        ) AS days_to_next,
        CASE
            WHEN next_admittime IS NULL THEN 0
            WHEN DATE_DIFF('day',
                CAST(dischtime AS TIMESTAMP),
                CAST(next_admittime AS TIMESTAMP)
            ) <= 30 THEN 1
            ELSE 0
        END AS readmitted_30d
    FROM all_admissions
)
SELECT
    subject_id,
    hadm_id,
    gender,
    age_at_admission,
    admission_type,
    insurance,
    race,
    discharge_location,
    marital_status,
    los_days,
    came_via_ed,
    prior_admissions_count,
    days_to_next,
    readmitted_30d
FROM with_target
"""

df_target = con.execute(target_query).df()

print(f"Target table shape: {df_target.shape}")
print(f"\nTarget distribution:")
print(df_target['readmitted_30d'].value_counts())
print(f"\nReadmission rate: {df_target['readmitted_30d'].mean()*100:.1f}%")
print(df_target.head())

Target table shape: (24069, 14)

Target distribution:
readmitted_30d
0    19236
1     4833
Name: count, dtype: int64

Readmission rate: 20.1%
   subject_id   hadm_id gender  age_at_admission  admission_type insurance  \
0    10014142  28915431      M                37  EU OBSERVATION  Medicaid   
1    10044085  23911180      F                28        EW EMER.  Medicaid   
2    10044085  22847862      F                29        EW EMER.  Medicaid   
3    10046166  20474438      M                67        EW EMER.   Private   
4    10046166  23223084      M                68        ELECTIVE   Private   

    race discharge_location marital_status  los_days  came_via_ed  \
0  WHITE               None         SINGLE         0            1   
1  WHITE               HOME         SINGLE         1            1   
2  WHITE     AGAINST ADVICE         SINGLE         0            1   
3  WHITE               HOME        MARRIED         4            1   
4  WHITE               HOME        MARRIED  

# Section 3: Join Target + CCSR Features into One Modeling Table


In [7]:

# The CCSR table has one row per patient (subject_id level)
# The target table has one row per admission (hadm_id level)
# We merge on subject_id — each admission gets the patient's CCSR flags

df_model = df_target.merge(ccsr, on='subject_id', how='left')

print(f"Modeling table shape: {df_model.shape}")
print(f"Columns: {len(df_model.columns)} total")
print(f"\nTarget distribution:")
print(df_model['readmitted_30d'].value_counts())
print(f"\nReadmission rate: {df_model['readmitted_30d'].mean()*100:.1f}%")
print(f"\nMissing values (top 10):")
print(df_model.isnull().sum().sort_values(ascending=False).head(10))
print(f"\nFirst 3 rows:")
print(df_model.head(3))

Modeling table shape: (24069, 719)
Columns: 719 total

Target distribution:
readmitted_30d
0    19236
1     4833
Name: count, dtype: int64

Readmission rate: 20.1%

Missing values (top 10):
days_to_next          9764
discharge_location    6744
marital_status         504
insurance              387
subject_id               0
'INJ054'                 0
'INJ039'                 0
'INJ040'                 0
'INJ041'                 0
'INJ042'                 0
dtype: int64

First 3 rows:
   subject_id   hadm_id gender  age_at_admission  admission_type insurance  \
0    10014142  28915431      M                37  EU OBSERVATION  Medicaid   
1    10044085  23911180      F                28        EW EMER.  Medicaid   
2    10044085  22847862      F                29        EW EMER.  Medicaid   

    race discharge_location marital_status  los_days  ...  'SYM010'  'SYM011'  \
0  WHITE               None         SINGLE         0  ...         0         0   
1  WHITE               HOME         S

In [8]:
df_model.head(10)
#df_model.describe()

,subject_id,hadm_id,gender,age_at_admission,admission_type,insurance,race,discharge_location,marital_status,los_days,...,'SYM010','SYM011','SYM012','SYM013','SYM014','SYM015','SYM016','SYM017','SYM018','XXX000'
0,10014142,28915431,M,37,EU OBSERVATION,Medicaid,WHITE,None,SINGLE,0,...,0,0,0,0,1,0,0,0,0,0
1,10044085,23911180,F,28,EW EMER.,Medicaid,WHITE,HOME,SINGLE,1,...,0,0,0,0,0,0,0,0,0,0
2,10044085,22847862,F,29,EW EMER.,Medicaid,WHITE,AGAINST ADVICE,SINGLE,0,...,0,0,0,0,0,0,0,0,0,0
3,10046166,20474438,M,67,EW EMER.,Private,WHITE,HOME,MARRIED,4,...,0,0,0,0,0,0,0,0,0,0
4,10046166,23223084,M,68,ELECTIVE,Private,WHITE,HOME,MARRIED,2,...,0,0,0,0,0,0,0,0,0,0
5,10046166,29381313,M,68,DIRECT EMER.,Private,WHITE,HOME,MARRIED,1,...,0,0,0,0,0,0,0,0,0,0
6,10046166,25512766,M,68,DIRECT OBSERVATION,Private,WHITE,None,MARRIED,3,...,0,0,0,0,0,0,0,0,0,0
7,10046166,28160175,M,68,EU OBSERVATION,Private,WHITE,None,MARRIED,1,...,0,0,0,0,0,0,0,0,0,0
8,10046166,22857894,M,68,EU OBSERVATION,Private,OTHER,None,MARRIED,1,...,0,0,0,0,0,0,0,0,0,0
9,10046166,27653344,M,68,EW EMER.,Private,OTHER,HOSPICE,MARRIED,6,...,0,0,0,0,0,0,0,0,0,0
